# Reflection & Critic Workflow Test Suite

Tests RAG reflection, SQL reflection, draft refinement execution, and max loop protection.

In [ ]:
import sys
from pathlib import Path

# Resolve project root
cwd = Path.cwd().resolve()
if cwd.name == "tests":
    project_root = cwd.parent.parent
elif cwd.name == "backend":
    project_root = cwd.parent
else:
    project_root = cwd

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from backend.app.agents.graph.workflow import create_workflow
from backend.app.agents.graph.nodes.critic import critic_node
from backend.app.agents.graph.nodes.refine import refine_node
from backend.app.llm.groq_provider import GroqProvider
from backend.app.llm.llm_client import LLMClient

provider = GroqProvider()
llm_client = LLMClient(provider)
workflow = create_workflow(llm_client)
print("Reflection/Critic Workflow compiled successfully!")

In [ ]:
# Test 1 — RAG Reflection (Draft -> Critic -> Final)
res_rag = await workflow.ainvoke({"user_message": "How many annual leave days does NexaTech provide?"})
print("RAG Reflection Result:", res_rag)
assert res_rag["route"] == "rag"
assert "draft_response" in res_rag
assert "critic_approved" in res_rag
assert len(res_rag["final_response"]) > 0

In [ ]:
# Test 2 — SQL Reflection (SQL -> Result -> Draft -> Critic -> Final)
res_sql = await workflow.ainvoke({"user_message": "What is the total revenue in the sales database?"})
print("SQL Reflection Result:", res_sql)
assert res_sql["route"] == "sql"
assert "sql_result" in res_sql
assert "draft_response" in res_sql
assert "critic_approved" in res_sql
assert len(res_sql["final_response"]) > 0

In [ ]:
# Test 3 — Draft Refinement Execution
state_with_issue = {
    "user_message": "What is the annual leave allowance?",
    "rag_context": "Full-time employees receive 24 paid annual leave days.",
    "draft_response": "Employees get 10 days of leave.", # Incomplete draft
    "critic_reason": "Inaccurate leave count",
    "critic_suggestions": "Correct leave count to 24 paid annual leave days per context",
    "reflection_count": 0,
}
refined_state = await refine_node(state_with_issue, llm_client)
print("Refined State:", refined_state)
assert "24" in refined_state["draft_response"]
assert refined_state["reflection_count"] == 1

In [ ]:
# Test 4 — Loop Protection Check (Max attempts = 1)
from backend.app.agents.graph.workflow import MAX_REFLECTION_ATTEMPTS

state_max_count = {
    "critic_approved": False,
    "reflection_count": 1, # Already reached max attempt
}

# Evaluate routing condition logic
approved = state_max_count.get("critic_approved", True)
count = state_max_count.get("reflection_count", 0)
next_node = "refine_node" if (not approved and count < MAX_REFLECTION_ATTEMPTS) else "final_response_node"

print(f"Next node when count=1 and approved=False: '{next_node}'")
assert next_node == "final_response_node" # Must NOT loop to refine_node